# 🚕 TaaSim — Week 1 : Exploration des données Porto
**ENSA Al Hoceima · Advanced Big Data · 2025–2026**

---

## 🗺️ Objectifs de ce notebook

| Étape | Tâche | Livrable |
|-------|-------|---------|
| 0 | Connexion à MinIO et chargement du CSV Porto | Dataset chargé en Spark |
| 1 | Exploration du schéma (types, taille, colonnes) | Description du dataset |
| 2 | Nettoyage (valeurs manquantes, POLYLINE vides) | Dataset propre |
| 3 | Distribution des durées de trajet | Histogramme |
| 4 | Analyse par CALL_TYPE (A/B/C) | Camembert + tableau |
| 5 | Courbe de demande temporelle | Courbe heure/jour |
| 6 | Zone remapping Porto → Casablanca | DataFrame enrichi |
| 7 | Visualisation OSM Casablanca | Carte folium |
| 8 | Démarrage Kafka producers | Vérification topics |

---

## ⚙️ Architecture rappel

```
MinIO (raw/porto-trips/) → PySpark → Exploration & Cleaning → MinIO (curated/)
                                                ↓
                                     Zone Remapping CSV
                                                ↓
                                     Kafka Producers (simulation)
```

---
## ÉTAPE 0 — Imports & Connexion MinIO + SparkSession

### 💡 Explication
Avant tout, on initialise :
- **PySpark** : moteur de traitement distribué. `SparkSession` est le point d'entrée unique.
- **S3A connector** : permet à Spark de lire/écrire dans MinIO comme si c'était un bucket S3 AWS.
- **boto3** : client Python pour MinIO (vérification des buckets).

### ⚠️ Points critiques
- `fs.s3a.endpoint` doit pointer vers `http://minio:9000` (nom Docker interne, pas localhost).
- `fs.s3a.path.style.access=true` est **obligatoire** avec MinIO (contrairement à AWS S3).
- Les JARs `hadoop-aws` et `aws-java-sdk-bundle` doivent être dans le classpath Spark.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 0 — Imports, SparkSession, connexion MinIO
# ─────────────────────────────────────────────────────────

import os
import json
import math
import boto3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import folium
from folium.plugins import HeatMap
from datetime import datetime
from botocore.client import Config

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType,
    BooleanType, ArrayType, DoubleType
)

# ── Configuration globale ──────────────────────────────────
MINIO_ENDPOINT   = "http://minio:9000"     # Nom réseau Docker
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin123"
MINIO_BUCKET_RAW = "raw"
PORTO_PREFIX     = "porto-trips/"

# ── Initialisation SparkSession avec connecteur S3A ────────
spark = (
    SparkSession.builder
    .appName("TaaSim-W1-Porto-Exploration")
    .master("spark://spark-master:7077")    # Cluster Docker
    # — Connecteur MinIO via S3A
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",              "org.apache.hadoop.fs.s3a.S3AFileSystem")
    # — Performance
    .config("spark.sql.adaptive.enabled",            "true")
    .config("spark.sql.shuffle.partitions",          "8")   # Adapté à 1 worker
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")  # Moins de logs verbeux
print(f"Spark version : {spark.version}")
print(f"SparkSession créée — App: {spark.sparkContext.appName}")

# ── Vérification connexion MinIO avec boto3 ────────────────
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4")
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print(f"\n Buckets MinIO disponibles : {buckets}")

# Vérifier que le fichier Porto est bien uploadé
try:
    objects = s3.list_objects_v2(Bucket=MINIO_BUCKET_RAW, Prefix=PORTO_PREFIX)
    files   = [o["Key"] for o in objects.get("Contents", [])]
    print(f" Fichiers Porto dans MinIO: {files}")
except Exception as e:
    print(f"  Erreur : {e}")
    print("   → Uploadez d'abord le CSV Porto dans MinIO raw/porto-trips/")

---
## ÉTAPE 1 — Définition du schéma et chargement du CSV

### 💡 Explication
Le dataset Porto contient **~1.7 million de lignes**. Chaque ligne = **1 trajet taxi complet**.

| Colonne | Type | Description |
|---------|------|-------------|
| `TRIP_ID` | String | Identifiant unique du trajet |
| `CALL_TYPE` | String | Mode de hèlement : A=dispatch central, B=station taxi, C=rue |
| `ORIGIN_CALL` | String | ID du client (si CALL_TYPE=A, sinon NULL) |
| `ORIGIN_STAND` | String | ID de la station (si CALL_TYPE=B, sinon NULL) |
| `TAXI_ID` | Long | Identifiant du taxi (442 taxis) |
| `TIMESTAMP` | Long | Heure de départ en **Unix epoch seconds** |
| `DAY_TYPE` | String | A=jour normal, B=jour férié, C=veille de férié |
| `MISSING_DATA` | Boolean | True = GPS manquant sur ce trajet |
| `POLYLINE` | String | **JSON array** de [lon, lat] toutes les **15 secondes** |

### ⚠️ Piège de débutant
- `POLYLINE` est une **chaîne de caractères** contenant du JSON, pas un tableau natif → il faut la parser avec `from_json()`.
- `TIMESTAMP` est en **secondes Unix** → convertir avec `F.from_unixtime()` pour avoir une date lisible.
- Définir le schéma **manuellement** est bien plus rapide que `inferSchema=True` sur 1.7M lignes.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 1 — Schéma explicite + chargement CSV depuis MinIO
# ─────────────────────────────────────────────────────────

# Schéma Porto — défini manuellement pour performance
porto_schema = StructType([
    StructField("TRIP_ID",      StringType(),  nullable=False),
    StructField("CALL_TYPE",    StringType(),  nullable=False),
    StructField("ORIGIN_CALL",  StringType(),  nullable=True),  # NULL si B ou C
    StructField("ORIGIN_STAND", StringType(),  nullable=True),  # NULL si A ou C
    StructField("TAXI_ID",      LongType(),    nullable=False),
    StructField("TIMESTAMP",    LongType(),    nullable=False),  # Unix epoch
    StructField("DAY_TYPE",     StringType(),  nullable=False),
    StructField("MISSING_DATA", BooleanType(), nullable=False),
    StructField("POLYLINE",     StringType(),  nullable=True),  # JSON string
])

# Chargement depuis MinIO
PORTO_S3_PATH = f"s3a://{MINIO_BUCKET_RAW}/{PORTO_PREFIX}train.csv"

df_raw = (
    spark.read
    .option("header",      "true")
    .option("quote",       '"')     # Le CSV Porto utilise des guillemets pour POLYLINE
    .option("escape",      '"')
    .schema(porto_schema)
    .csv(PORTO_S3_PATH)
)

# Mise en cache — on va faire beaucoup d'opérations sur ce dataset
df_raw.cache()

total_rows = df_raw.count()
print(f" Dataset Porto chargé : {total_rows:,} lignes")
print(f"   Partitions Spark    : {df_raw.rdd.getNumPartitions()}")

# Aperçu des 5 premières lignes
print("\n📋 Aperçu du dataset (5 premières lignes) :")
df_raw.show(5, truncate=60)

---
## ÉTAPE 2 — Exploration du schéma et valeurs manquantes

### 💡 Explication
Avant tout traitement, il faut **auditer la qualité des données** :
- Combien de trajets ont `MISSING_DATA=True` ? → Ces trajets ont des trous GPS → **à exclure**.
- Des POLYLINE vides `[]` ? → Trajet sans aucun point GPS → **à exclure**.
- Distribution de `CALL_TYPE` → comprendre comment les taxis sont hélés à Porto (et donc à Casablanca par analogie).
- Distribution de `DAY_TYPE` → majorité de jours normaux (A) attendue.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 2 — Audit qualité des données
# ─────────────────────────────────────────────────────────

print("═" * 60)
print("  AUDIT QUALITÉ — DATASET PORTO")
print("═" * 60)

# 2a. Schéma complet
print("\nSchéma Spark :")
df_raw.printSchema()

# 2b. Statistiques descriptives des colonnes numériques
print("\nStatistiques descriptives :")
df_raw.select("TAXI_ID", "TIMESTAMP").describe().show()

# 2c. Compte des valeurs nulles par colonne
print("\n Valeurs nulles par colonne :")
null_counts = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])
null_counts.show()

# 2d. Trajets avec données GPS manquantes
missing_gps   = df_raw.filter(F.col("MISSING_DATA") == True).count()
empty_polyline = df_raw.filter(
    F.col("POLYLINE").isNull() | (F.col("POLYLINE") == "[]") | (F.col("POLYLINE") == "")
).count()

print(f"\nTrajets avec MISSING_DATA=True  : {missing_gps:,} ({missing_gps/total_rows*100:.2f}%)")
print(f" Trajets avec POLYLINE vide/null  : {empty_polyline:,} ({empty_polyline/total_rows*100:.2f}%)")

# 2e. Distribution DAY_TYPE
print("\n Distribution DAY_TYPE :")
df_raw.groupBy("DAY_TYPE").count().orderBy("count", ascending=False).show()

# 2f. Nombre de taxis uniques
n_taxis = df_raw.select("TAXI_ID").distinct().count()
print(f"\nNombre de taxis uniques : {n_taxis} (attendu: ~442)")

# 2g. Plage temporelle du dataset
ts_stats = df_raw.select(
    F.min("TIMESTAMP").alias("ts_min"),
    F.max("TIMESTAMP").alias("ts_max")
).collect()[0]

date_min = datetime.utcfromtimestamp(ts_stats["ts_min"]).strftime("%Y-%m-%d")
date_max = datetime.utcfromtimestamp(ts_stats["ts_max"]).strftime("%Y-%m-%d")
print(f"\n Plage temporelle : {date_min} → {date_max}")

---
## ÉTAPE 3 — Nettoyage et enrichissement du dataset

### 💡 Explication des transformations
On crée `df_clean` qui sera la base de toutes les analyses suivantes.

| Transformation | Raison |
|---------------|--------|
| Supprimer `MISSING_DATA=True` | GPS incomplet → durée et coordonnées faussées |
| Supprimer `POLYLINE` vide/null | Aucune donnée géographique → inutilisable |
| Parser `POLYLINE` JSON → Array | Nécessaire pour extraire coordonnées |
| Calculer `trip_duration_sec` | `len(polyline) * 15` secondes (1 point = 15 sec) |
| Convertir `TIMESTAMP` → `datetime` | Lisibilité et extraction heure/jour |
| Ajouter `hour_of_day`, `day_of_week` | Features pour ML et analyses temporelles |
| Extraire `origin_lon/lat`, `dest_lon/lat` | Premier et dernier point du POLYLINE |

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 3 — Nettoyage et enrichissement
# ─────────────────────────────────────────────────────────

# Schéma du POLYLINE : tableau de [lon, lat]
polyline_schema = ArrayType(ArrayType(DoubleType()))

df_clean = (
    df_raw
    # ── 1. Exclure trajets avec GPS manquant ──────────────
    .filter(F.col("MISSING_DATA") == False)
    
    # ── 2. Exclure POLYLINE vides ─────────────────────────
    .filter(
        F.col("POLYLINE").isNotNull() &
        (F.col("POLYLINE") != "[]") &
        (F.col("POLYLINE") != "")
    )
    
    # ── 3. Parser le POLYLINE JSON → ArrayType ────────────
    .withColumn("polyline_parsed", F.from_json(F.col("POLYLINE"), polyline_schema))
    
    # ── 4. Calculer la durée : nb_points × 15 secondes ───
    .withColumn("n_gps_points",      F.size("polyline_parsed"))
    .withColumn("trip_duration_sec", F.col("n_gps_points") * 15)
    .withColumn("trip_duration_min", F.round(F.col("trip_duration_sec") / 60.0, 2))
    
    # ── 5. Convertir TIMESTAMP Unix → datetime Spark ──────
    .withColumn("event_datetime", F.from_unixtime(F.col("TIMESTAMP")))
    .withColumn("event_datetime", F.to_timestamp("event_datetime"))
    
    # ── 6. Extraire features temporelles ─────────────────
    .withColumn("hour_of_day",  F.hour("event_datetime"))
    .withColumn("day_of_week",  F.dayofweek("event_datetime"))  # 1=Dimanche, 7=Samedi
    .withColumn("month",        F.month("event_datetime"))
    .withColumn("is_weekend",   F.col("day_of_week").isin([1, 7]))
    
    # ── 7. Extraire coordonnées origine et destination ────
    # Premier point du polyline = origine
    .withColumn("origin_lon", F.col("polyline_parsed")[0][0])
    .withColumn("origin_lat", F.col("polyline_parsed")[0][1])
    # Dernier point = destination
    .withColumn("dest_lon", F.col("polyline_parsed")[F.col("n_gps_points") - 1][0])
    .withColumn("dest_lat", F.col("polyline_parsed")[F.col("n_gps_points") - 1][1])
    
    # ── 8. Supprimer les trajets trop courts (<60 sec)
    #        ou trop longs (>2h = aberrant)
    .filter((F.col("trip_duration_sec") >= 60) & (F.col("trip_duration_sec") <= 7200))
)

df_clean.cache()
clean_count = df_clean.count()
removed = total_rows - clean_count

print(f"✅ Dataset nettoyé    : {clean_count:,} lignes")
print(f"🗑️  Lignes supprimées  : {removed:,} ({removed/total_rows*100:.2f}%)")
print("\n📋 Aperçu après nettoyage :")
df_clean.select(
    "TRIP_ID", "CALL_TYPE", "TAXI_ID",
    "event_datetime", "hour_of_day", "day_of_week",
    "trip_duration_min", "n_gps_points",
    "origin_lon", "origin_lat"
).show(5, truncate=30)

---
## ÉTAPE 4 — Distribution des durées de trajet

### 💡 Explication
La distribution des durées est critique pour TaaSim :
- Elle valide que les données sont **réalistes** (pic entre 5–20 min attendu pour une ville comme Porto ou Casablanca).
- Elle définit les **tranches de durée** à utiliser dans le ML et pour les SLA (ETA < 5 min pour le match).
- Les valeurs extrêmes (> 90 min) représentent probablement des courses aéroport ou des erreurs.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 4 — Distribution des durées de trajet
# ─────────────────────────────────────────────────────────

# Collecte des durées en Pandas pour visualisation
# Approx quantile en Spark (évite de tout collecter)
quantiles = df_clean.stat.approxQuantile(
    "trip_duration_min",
    [0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
    relativeError=0.01
)

labels = ["P5", "Q1", "Médiane", "Q3", "P90", "P95", "P99"]
print("📊 Distribution des durées de trajet (minutes) :")
print("-" * 40)
for lbl, val in zip(labels, quantiles):
    print(f"  {lbl:<10} : {val:.1f} min")

avg_dur = df_clean.select(F.mean("trip_duration_min")).collect()[0][0]
print(f"  {'Moyenne':<10} : {avg_dur:.1f} min")

# Histogramme via agrégation Spark (sans tout collecter)
# Créer des buckets de 5 minutes
df_hist = (
    df_clean
    .withColumn("bucket_5min", (F.col("trip_duration_min") / 5).cast("int") * 5)
    .filter(F.col("bucket_5min") <= 90)   # Cap à 90 min pour le graphe
    .groupBy("bucket_5min")
    .count()
    .orderBy("bucket_5min")
    .toPandas()
)

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Distribution des durées de trajet — Porto Taxi (proxy Casablanca)",
             fontsize=14, fontweight="bold", y=1.02)

# ── Histogramme buckets
ax1 = axes[0]
bars = ax1.bar(df_hist["bucket_5min"], df_hist["count"] / 1000,
               width=4, color="#1565C0", edgecolor="white", linewidth=0.5)
ax1.axvline(x=avg_dur, color="#FF6F00", linestyle="--", linewidth=2,
            label=f"Moyenne : {avg_dur:.1f} min")
ax1.axvline(x=quantiles[2], color="#2E7D32", linestyle="-", linewidth=2,
            label=f"Médiane : {quantiles[2]:.1f} min")
ax1.set_xlabel("Durée du trajet (minutes)", fontsize=11)
ax1.set_ylabel("Nombre de trajets (milliers)", fontsize=11)
ax1.set_title("Histogramme des durées (buckets 5 min)", fontsize=12)
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# ── Box plot catégoriel par CALL_TYPE
df_box = (
    df_clean
    .select("CALL_TYPE", "trip_duration_min")
    .filter(F.col("trip_duration_min") <= 60)
    .sample(fraction=0.1)   # 10% pour collecte rapide
    .toPandas()
)

ax2 = axes[1]
call_type_labels = {"A": "Dispatch\nCentral (A)", "B": "Station\nTaxi (B)", "C": "Rue\n(Hèlement C)"}
df_box["CALL_TYPE_label"] = df_box["CALL_TYPE"].map(call_type_labels)

colors = ["#1565C0", "#2E7D32", "#FF6F00"]
for i, (ct, grp) in enumerate(df_box.groupby("CALL_TYPE")):
    ax2.boxplot(grp["trip_duration_min"].dropna(),
                positions=[i], widths=0.5,
                patch_artist=True,
                boxprops=dict(facecolor=colors[i], alpha=0.7),
                medianprops=dict(color="white", linewidth=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))

ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(["Dispatch Central\n(A)", "Station Taxi\n(B)", "Rue — Hèlement\n(C)"])
ax2.set_ylabel("Durée du trajet (minutes)", fontsize=11)
ax2.set_title("Durée par mode de hèlement (CALL_TYPE)", fontsize=12)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/porto_durations.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Graphique sauvegardé dans /tmp/porto_durations.png")

---
## ÉTAPE 5 — Analyse par CALL_TYPE

### 💡 Explication — Lien avec Casablanca
C'est la colonne la plus importante pour la **pertinence du mapping** Porto → Casablanca :

| CALL_TYPE Porto | Équivalent Casablanca |
|----------------|----------------------|
| **A** — Dispatch central | Application TaaSim (réservation digitale) |
| **B** — Station taxi | Stations de petits taxis (Mers Sultan, Guéliz...) |
| **C** — Hèlement rue | Grand taxi / petit taxi hélé sur le trottoir |

Ce mapping justifie **pourquoi Porto est un bon proxy** pour Casablanca, comme indiqué dans le cahier des charges.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 5 — Analyse par CALL_TYPE
# ─────────────────────────────────────────────────────────

# Comptage par CALL_TYPE
df_calltype = (
    df_clean
    .groupBy("CALL_TYPE")
    .agg(
        F.count("*").alias("nb_trajets"),
        F.round(F.mean("trip_duration_min"), 1).alias("duree_moy_min"),
        F.round(F.mean("n_gps_points"), 0).alias("points_gps_moy")
    )
    .orderBy("CALL_TYPE")
    .toPandas()
)

df_calltype["pourcentage"] = (df_calltype["nb_trajets"] / df_calltype["nb_trajets"].sum() * 100).round(1)
df_calltype["equivalent_casa"] = [
    "App TaaSim (réservation digitale)",
    "Station de taxi (petit taxi)",
    "Hèlement rue (grand taxi)"
]

print("\n📊 Distribution par CALL_TYPE — Mapping Porto → Casablanca")
print("═" * 70)
print(df_calltype.to_string(index=False))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Analyse par CALL_TYPE — Porto Taxi", fontsize=14, fontweight="bold")

# ── Camembert
colors_pie = ["#1565C0", "#2E7D32", "#FF6F00"]
wedges, texts, autotexts = axes[0].pie(
    df_calltype["nb_trajets"],
    labels=[f"Type {ct}" for ct in df_calltype["CALL_TYPE"]],
    autopct="%1.1f%%",
    colors=colors_pie,
    startangle=90,
    textprops={"fontsize": 12}
)
for at in autotexts:
    at.set_fontweight("bold")
    at.set_color("white")

axes[0].set_title("Répartition des trajets par mode de hèlement", fontsize=11)

# Légende avec mapping Casablanca
legend_labels = [
    f"A ({df_calltype.loc[0,'pourcentage']}%) — {df_calltype.loc[0,'equivalent_casa']}",
    f"B ({df_calltype.loc[1,'pourcentage']}%) — {df_calltype.loc[1,'equivalent_casa']}",
    f"C ({df_calltype.loc[2,'pourcentage']}%) — {df_calltype.loc[2,'equivalent_casa']}"
]
axes[0].legend(wedges, legend_labels, loc="lower center",
               bbox_to_anchor=(0.5, -0.15), fontsize=9)

# ── Barres durée moyenne
bars = axes[1].bar(
    [f"Type {ct}" for ct in df_calltype["CALL_TYPE"]],
    df_calltype["duree_moy_min"],
    color=colors_pie, edgecolor="white", linewidth=0.5
)
for bar, val in zip(bars, df_calltype["duree_moy_min"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val} min", ha="center", va="bottom", fontweight="bold")

axes[1].set_ylabel("Durée moyenne du trajet (minutes)", fontsize=11)
axes[1].set_title("Durée moyenne par CALL_TYPE", fontsize=11)
axes[1].grid(axis="y", alpha=0.3)
axes[1].set_ylim(0, max(df_calltype["duree_moy_min"]) * 1.2)

plt.tight_layout()
plt.savefig("/tmp/porto_call_types.png", dpi=150, bbox_inches="tight")
plt.show()

---
## ÉTAPE 6 — Courbe de demande temporelle

### 💡 Explication
La courbe de demande est la **colonne vertébrale du ML** et du simulateur Kafka :
- Elle révèle les **heures de pointe** (morning rush, evening rush).
- Le `trip_request_producer.py` utilise cette courbe pour doser les événements Kafka.
- Les features `hour_of_day`, `is_weekend`, `is_friday` du modèle GBT viennent directement de cette analyse.

### 📍 Attendu pour Casablanca
- Pic fort **7h–9h** (rush matin vers centres d'affaires)
- Pic fort **17h–19h** (rush soir)
- Vendredi midi **12h–14h** atypique (prière du vendredi → baisse, puis reprise)

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 6 — Courbe de demande temporelle
# ─────────────────────────────────────────────────────────

# ── 6a. Demande par heure de la journée ───────────────────
df_hourly = (
    df_clean
    .groupBy("hour_of_day")
    .agg(F.count("*").alias("nb_trajets"))
    .orderBy("hour_of_day")
    .toPandas()
)

# ── 6b. Demande par jour de la semaine ────────────────────
# PySpark dayofweek : 1=Dimanche, 2=Lundi, ..., 7=Samedi
day_labels = {1: "Dim", 2: "Lun", 3: "Mar", 4: "Mer",
              5: "Jeu", 6: "Ven", 7: "Sam"}

df_daily = (
    df_clean
    .groupBy("day_of_week")
    .agg(F.count("*").alias("nb_trajets"))
    .orderBy("day_of_week")
    .toPandas()
)
df_daily["jour"] = df_daily["day_of_week"].map(day_labels)

# ── 6c. Heatmap heure × jour (demande horaire croisée) ────
df_heatmap = (
    df_clean
    .groupBy("day_of_week", "hour_of_day")
    .agg(F.count("*").alias("nb_trajets"))
    .toPandas()
)
heatmap_pivot = df_heatmap.pivot(index="day_of_week", columns="hour_of_day", values="nb_trajets")
heatmap_pivot.index = [day_labels.get(d, d) for d in heatmap_pivot.index]

# ── Visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle("Analyse temporelle de la demande — Porto Taxi\n(Proxy pour Casablanca)",
             fontsize=15, fontweight="bold")

# Graphe 1 — Demande horaire
ax = axes[0, 0]
ax.plot(df_hourly["hour_of_day"], df_hourly["nb_trajets"] / 1000,
        color="#1565C0", linewidth=2.5, marker="o", markersize=5)
ax.fill_between(df_hourly["hour_of_day"], df_hourly["nb_trajets"] / 1000,
                alpha=0.15, color="#1565C0")
# Annotations pointes
ax.axvspan(7, 9, alpha=0.15, color="#FF6F00", label="Rush matin (7h–9h)")
ax.axvspan(17, 19, alpha=0.15, color="#2E7D32", label="Rush soir (17h–19h)")
ax.set_xlabel("Heure de la journée", fontsize=11)
ax.set_ylabel("Nb de trajets (milliers)", fontsize=11)
ax.set_title("Demande par heure de la journée", fontsize=12)
ax.set_xticks(range(0, 24))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Graphe 2 — Demande journalière
ax = axes[0, 1]
bar_colors = ["#FF6F00" if d in [1, 7] else "#1565C0" for d in df_daily["day_of_week"]]
ax.bar(df_daily["jour"], df_daily["nb_trajets"] / 1000, color=bar_colors)
ax.set_ylabel("Nb de trajets (milliers)", fontsize=11)
ax.set_title("Demande par jour de la semaine", fontsize=12)
ax.grid(axis="y", alpha=0.3)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#1565C0", label="Semaine"),
    Patch(color="#FF6F00", label="Week-end")
], fontsize=9)

# Graphe 3 — Heatmap jour × heure
ax = axes[1, 0]
sns.heatmap(heatmap_pivot / 1000, ax=ax, cmap="YlOrRd",
            fmt=".0f", linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Milliers de trajets"})
ax.set_title("Heatmap : Jour × Heure (demande en milliers)", fontsize=12)
ax.set_xlabel("Heure de la journée", fontsize=10)
ax.set_ylabel("Jour de la semaine", fontsize=10)

# Graphe 4 — Demande mensuelle (saisonnalité)
df_monthly = (
    df_clean
    .groupBy("month")
    .count()
    .orderBy("month")
    .toPandas()
)
month_names = {1:"Jan",2:"Fév",3:"Mar",4:"Avr",5:"Mai",6:"Jun",
               7:"Jul",8:"Aoû",9:"Sep",10:"Oct",11:"Nov",12:"Déc"}
df_monthly["mois"] = df_monthly["month"].map(month_names)

ax = axes[1, 1]
ax.bar(df_monthly["mois"], df_monthly["count"] / 1000, color="#1565C0", alpha=0.8)
ax.set_ylabel("Nb de trajets (milliers)", fontsize=11)
ax.set_title("Demande mensuelle (Juil 2013 – Juin 2014)", fontsize=12)
ax.grid(axis="y", alpha=0.3)
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("/tmp/porto_temporal.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Export de la courbe horaire pour le Kafka producer ────
# Le trip_request_producer.py utilise ce vecteur pour doser les événements
hourly_weights = (
    df_hourly
    .set_index("hour_of_day")["nb_trajets"]
    .reindex(range(24), fill_value=0)
)
hourly_weights_normalized = (hourly_weights / hourly_weights.max()).round(4).to_dict()

with open("/tmp/hourly_demand_weights.json", "w") as f:
    json.dump(hourly_weights_normalized, f, indent=2)

print("✅ Poids horaires exportés → /tmp/hourly_demand_weights.json")
print("   (À utiliser dans trip_request_producer.py)")
print(f"\n📈 Top 3 heures de pointe :")
top3 = df_hourly.nlargest(3, "nb_trajets")
for _, row in top3.iterrows():
    print(f"   {int(row['hour_of_day']):02d}h : {int(row['nb_trajets']):,} trajets")

---
## ÉTAPE 7 — Zone Remapping : Porto → Casablanca

### 💡 Explication complète

C'est la **tâche data engineering centrale de la semaine 1**. Le problème :
- Porto est dans le coin `[-8.8, -8.5]°W × [41.1, 41.2]°N`
- Casablanca est dans le coin `[-7.7, -7.5]°W × [33.4, 33.7]°N`

La transformation est une **interpolation linéaire** :
```
lon_casa = lon_min_casa + (lon_porto - lon_min_porto) / (lon_max_porto - lon_min_porto)
                        × (lon_max_casa - lon_min_casa)
```

Ce qui est **préservé** après remapping :
- ✅ Distributions de durée de trajet
- ✅ Courbes de demande temporelle  
- ✅ Répartition CALL_TYPE
- ✅ Patterns de demande zone-à-zone

Ce qui est **distordu** :
- ❌ La topographie réelle (mer, autoroutes, quartiers)
- ❌ Les distances absolues

### 🗺️ Zones Casablanca (16 arrondissements)
Les 22 zones de Porto sont mappées vers les 16 arrondissements de Casablanca via `zone_mapping.csv`.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 7 — Zone Remapping Porto → Casablanca
# ─────────────────────────────────────────────────────────

# ── 7a. Bounding boxes ────────────────────────────────────
# Porto (source) — extrait des données réelles
PORTO_LON_MIN, PORTO_LON_MAX = -8.83, -8.43
PORTO_LAT_MIN, PORTO_LAT_MAX =  41.10,  41.25

# Casablanca (cible) — Cahier des charges §9.3
CASA_LON_MIN, CASA_LON_MAX = -7.70, -7.50
CASA_LAT_MIN, CASA_LAT_MAX =  33.45,  33.68

print("🗺️  Bounding Boxes")
print(f"   Porto      : lon [{PORTO_LON_MIN}, {PORTO_LON_MAX}]  lat [{PORTO_LAT_MIN}, {PORTO_LAT_MAX}]")
print(f"   Casablanca : lon [{CASA_LON_MIN}, {CASA_LON_MAX}]  lat [{CASA_LAT_MIN}, {CASA_LAT_MAX}]")

# ── 7b. Fonctions de remapping (UDF Spark) ────────────────

def remap_lon(lon_porto: float) -> float:
    """Remapping linéaire de la longitude Porto → Casablanca."""
    if lon_porto is None:
        return None
    ratio = (lon_porto - PORTO_LON_MIN) / (PORTO_LON_MAX - PORTO_LON_MIN)
    return round(CASA_LON_MIN + ratio * (CASA_LON_MAX - CASA_LON_MIN), 6)

def remap_lat(lat_porto: float) -> float:
    """Remapping linéaire de la latitude Porto → Casablanca."""
    if lat_porto is None:
        return None
    ratio = (lat_porto - PORTO_LAT_MIN) / (PORTO_LAT_MAX - PORTO_LAT_MIN)
    return round(CASA_LAT_MIN + ratio * (CASA_LAT_MAX - CASA_LAT_MIN), 6)

# Enregistrement des UDF Spark
udf_remap_lon = F.udf(remap_lon, DoubleType())
udf_remap_lat = F.udf(remap_lat, DoubleType())

# ── 7c. Table des 16 arrondissements de Casablanca ────────
# Source : découpage officiel Casablanca (simplifié pour le projet)
casablanca_zones = [
    # (zone_id, nom, lon_min, lon_max, lat_min, lat_max, type)
    (1,  "Anfa",              -7.650, -7.580, 33.570, 33.620, "residential"),
    (2,  "Aïn Chock",         -7.600, -7.520, 33.530, 33.580, "residential"),
    (3,  "Aïn Sebaâ",         -7.580, -7.510, 33.580, 33.640, "commercial"),
    (4,  "Al Fida",           -7.630, -7.580, 33.560, 33.600, "commercial"),
    (5,  "Ben Msik",          -7.560, -7.510, 33.540, 33.580, "residential"),
    (6,  "Bernoussi",         -7.560, -7.500, 33.600, 33.650, "residential"),
    (7,  "Bouskoura",         -7.600, -7.530, 33.440, 33.490, "residential"),
    (8,  "Centre-Ville",      -7.630, -7.590, 33.590, 33.620, "transit_hub"),
    (9,  "Derb Sultan",       -7.620, -7.570, 33.540, 33.580, "residential"),
    (10, "El Hank",           -7.660, -7.620, 33.570, 33.600, "commercial"),
    (11, "Hay Hassani",       -7.640, -7.580, 33.510, 33.560, "residential"),
    (12, "Hay Mohammadi",     -7.600, -7.540, 33.560, 33.600, "residential"),
    (13, "Mers Sultan",       -7.625, -7.590, 33.580, 33.610, "transit_hub"),
    (14, "Moulay Rachid",     -7.560, -7.510, 33.510, 33.560, "residential"),
    (15, "Sidi Belyout",      -7.620, -7.590, 33.590, 33.620, "commercial"),
    (16, "Sidi Moumen",       -7.540, -7.500, 33.560, 33.620, "residential"),
]

# Créer un DataFrame Spark des zones
zones_schema = StructType([
    StructField("zone_id",   IntegerType(), False),
    StructField("zone_name", StringType(),  False),
    StructField("lon_min",   DoubleType(),  False),
    StructField("lon_max",   DoubleType(),  False),
    StructField("lat_min",   DoubleType(),  False),
    StructField("lat_max",   DoubleType(),  False),
    StructField("zone_type", StringType(),  False),
])
df_zones = spark.createDataFrame(casablanca_zones, schema=zones_schema)
df_zones.cache()

print("\n🗂️  Zones Casablanca définies :")
df_zones.show(16, truncate=False)

# ── 7d. Calcul centroides de chaque zone ─────────────────
df_zones_centroids = df_zones.withColumn(
    "centroid_lon", (F.col("lon_min") + F.col("lon_max")) / 2
).withColumn(
    "centroid_lat", (F.col("lat_min") + F.col("lat_max")) / 2
)

# Exporter en CSV pour le starter kit
(
    df_zones_centroids.toPandas()
    .to_csv("/tmp/zone_mapping.csv", index=False)
)
print("✅ zone_mapping.csv exporté → /tmp/zone_mapping.csv")

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 7 (suite) — Application du remapping sur df_clean
# ─────────────────────────────────────────────────────────

# ── 7e. Appliquer le remapping sur les coordonnées ────────
df_remapped = (
    df_clean
    .withColumn("casa_origin_lon", udf_remap_lon(F.col("origin_lon")))
    .withColumn("casa_origin_lat", udf_remap_lat(F.col("origin_lat")))
    .withColumn("casa_dest_lon",   udf_remap_lon(F.col("dest_lon")))
    .withColumn("casa_dest_lat",   udf_remap_lat(F.col("dest_lat")))
)

# ── 7f. Assignation de zone par bounding box (cross join) ─
# Pour chaque trip, trouver quelle zone Casablanca contient l'origine
df_with_zones = (
    df_remapped
    .crossJoin(df_zones_centroids.select(
        "zone_id", "zone_name", "zone_type",
        "lon_min", "lon_max", "lat_min", "lat_max",
        "centroid_lon", "centroid_lat"
    ))
    .filter(
        (F.col("casa_origin_lon") >= F.col("lon_min")) &
        (F.col("casa_origin_lon") <  F.col("lon_max")) &
        (F.col("casa_origin_lat") >= F.col("lat_min")) &
        (F.col("casa_origin_lat") <  F.col("lat_max"))
    )
    .drop("lon_min", "lon_max", "lat_min", "lat_max")
)

df_with_zones.cache()
zoned_count = df_with_zones.count()
print(f"\n✅ Trajets avec zone assignée : {zoned_count:,}")
print(f"   Sur {clean_count:,} total → couverture : {zoned_count/clean_count*100:.1f}%")

# ── 7g. Distribution des trajets par zone ─────────────────
print("\n📊 Top 10 zones par volume de départs :")
(
    df_with_zones
    .groupBy("zone_id", "zone_name", "zone_type")
    .count()
    .orderBy(F.col("count").desc())
    .show(10)
)

# ── 7h. Sauvegarder dans MinIO curated/ ──────────────────
(
    df_with_zones
    .select(
        "TRIP_ID", "CALL_TYPE", "TAXI_ID",
        "event_datetime", "hour_of_day", "day_of_week", "month", "is_weekend",
        "trip_duration_min", "n_gps_points",
        "casa_origin_lon", "casa_origin_lat",
        "casa_dest_lon", "casa_dest_lat",
        "zone_id", "zone_name", "zone_type",
        "centroid_lon", "centroid_lat"
    )
    .write
    .mode("overwrite")
    .parquet("s3a://curated/porto-trips/")
)
print("\n✅ Dataset enrichi sauvegardé → s3a://curated/porto-trips/ (Parquet)")

---
## ÉTAPE 8 — Visualisation sur carte OSM (Folium)

### 💡 Explication
Folium génère des cartes interactives OpenStreetMap en HTML.
- On trace les **centroides des 16 zones** de Casablanca.
- On trace un **échantillon de points d'origine** des trajets remappés.
- La heatmap montre où se concentre la demande dans Casablanca.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 8 — Carte OSM Casablanca avec Folium
# ─────────────────────────────────────────────────────────

# Centre de Casablanca
CASA_CENTER = [33.5731, -7.5898]

m = folium.Map(
    location=CASA_CENTER,
    zoom_start=12,
    tiles="OpenStreetMap"
)

# ── Ajouter les 16 zones Casablanca ──────────────────────
zones_pd = df_zones_centroids.toPandas()

zone_colors = {
    "transit_hub":  "red",
    "commercial":   "orange",
    "residential":  "blue"
}

for _, row in zones_pd.iterrows():
    color = zone_colors.get(row["zone_type"], "gray")
    folium.CircleMarker(
        location=[row["centroid_lat"], row["centroid_lon"]],
        radius=15,
        popup=folium.Popup(
            f"<b>Zone {row['zone_id']} — {row['zone_name']}</b><br>"
            f"Type: {row['zone_type']}<br>"
            f"Centroïde: ({row['centroid_lat']:.4f}, {row['centroid_lon']:.4f})",
            max_width=250
        ),
        tooltip=f"Zone {row['zone_id']}: {row['zone_name']}",
        color="white",
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=2
    ).add_to(m)
    
    # Label de la zone
    folium.Marker(
        location=[row["centroid_lat"] + 0.003, row["centroid_lon"]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:8px;font-weight:bold;color:#333;">Z{row["zone_id"]}</div>',
            icon_size=(20, 10)
        )
    ).add_to(m)

# ── Heatmap des points d'origine (échantillon 5000 pts) ──
sample_origins = (
    df_with_zones
    .select("casa_origin_lat", "casa_origin_lon")
    .sample(fraction=0.003)   # ~5000 points sur 1.7M
    .toPandas()
    .dropna()
)

heat_data = list(zip(sample_origins["casa_origin_lat"],
                     sample_origins["casa_origin_lon"]))

HeatMap(
    heat_data,
    min_opacity=0.3,
    max_zoom=14,
    radius=15,
    blur=15,
    name="Densité départs"
).add_to(m)

# ── Légende ──────────────────────────────────────────────
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:999;
            background:white;padding:12px;border-radius:8px;
            border:2px solid #ccc;font-size:12px;">
  <b>🚕 TaaSim — Zones Casablanca</b><br>
  <span style="color:red;">●</span> Nœud transit<br>
  <span style="color:orange;">●</span> Commercial<br>
  <span style="color:blue;">●</span> Résidentiel<br>
  <span style="background:linear-gradient(to right,blue,red);display:inline-block;width:50px;height:8px;"></span> Densité départs
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Légende CALL_TYPE
folium.LayerControl().add_to(m)

MAP_PATH = "/tmp/casablanca_zones_map.html"
m.save(MAP_PATH)
print(f"✅ Carte sauvegardée → {MAP_PATH}")
print("   Ouvrir dans un navigateur pour voir la carte interactive")
print(f"   Nb points tracés sur la heatmap : {len(heat_data):,}")

# Afficher directement dans Jupyter si possible
try:
    from IPython.display import IFrame
    display(IFrame(MAP_PATH, width=900, height=500))
except:
    m

---
## ÉTAPE 9 — Démarrage des Kafka Producers

### 💡 Explication
Les producers sont des scripts Python qui **simulent** les flux temps réel :
- `vehicle_gps_producer.py` → rejoue les POLYLINE Porto sur le topic `raw.gps`
- `trip_request_producer.py` → génère des réservations sur `raw.trips`

Ici on crée et vérifie les topics Kafka, puis on teste la connexion avec `kafka-python`.

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 9a — Création des topics Kafka
# ─────────────────────────────────────────────────────────

from kafka import KafkaAdminClient, KafkaProducer, KafkaConsumer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError
import time, uuid

KAFKA_BROKER = "kafka:9092"

TOPICS = [
    {"name": "raw.gps",            "partitions": 3, "replication": 1},
    {"name": "raw.trips",          "partitions": 3, "replication": 1},
    {"name": "processed.gps",      "partitions": 3, "replication": 1},
    {"name": "processed.demand",   "partitions": 1, "replication": 1},
    {"name": "processed.matches",  "partitions": 1, "replication": 1},
]

admin = KafkaAdminClient(
    bootstrap_servers=KAFKA_BROKER,
    client_id="taasim-admin"
)

new_topics = [
    NewTopic(
        name=t["name"],
        num_partitions=t["partitions"],
        replication_factor=t["replication"],
        topic_configs={"retention.ms": str(7 * 24 * 3600 * 1000)}  # 7 jours
    )
    for t in TOPICS
]

try:
    admin.create_topics(new_topics=new_topics, validate_only=False)
    print("✅ Topics Kafka créés")
except TopicAlreadyExistsError:
    print("ℹ️  Topics déjà existants — OK")
except Exception as e:
    print(f"⚠️  Erreur création topics : {e}")

# Lister les topics
topics = admin.list_topics()
taasim_topics = [t for t in topics if not t.startswith("__")]
print(f"\n📋 Topics TaaSim actifs : {sorted(taasim_topics)}")
admin.close()

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 9b — GPS Producer (version test — 20 événements)
# ─────────────────────────────────────────────────────────
# Ceci est une version simplifiée pour vérifier la connexion Kafka.
# Le vrai producer (vehicle_gps_producer.py) est présenté dans l'étape 9c.

import json, random, time
from datetime import datetime, timezone

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: str(k).encode("utf-8")
)

# Test : envoyer 20 événements GPS fictifs
print("📡 Envoi de 20 événements GPS de test sur raw.gps...")

for i in range(20):
    taxi_id = random.randint(1, 442)
    zone    = random.randint(1, 16)
    
    # Coordonnées dans la bounding box Casablanca
    lat = round(random.uniform(CASA_LAT_MIN, CASA_LAT_MAX), 6)
    lon = round(random.uniform(CASA_LON_MIN, CASA_LON_MAX), 6)
    
    event = {
        "taxi_id":   taxi_id,
        "timestamp": int(datetime.now(timezone.utc).timestamp()),
        "lat":       lat,
        "lon":       lon,
        "speed":     round(random.uniform(0, 80), 1),
        "status":    random.choice(["available", "assigned"]),
        "zone_id":   zone
    }
    
    producer.send("raw.gps", key=taxi_id, value=event)
    print(f"  ✉️  taxi={taxi_id:03d} | zone={zone:02d} | lat={lat} | lon={lon}")

producer.flush()
producer.close()
print("\n✅ 20 événements envoyés sur raw.gps")

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 9c — Code complet : vehicle_gps_producer.py
# ─────────────────────────────────────────────────────────
# Ce code est le vrai producer à sauvegarder comme script Python séparé.
# Il rejoue les polylines Porto → Casablanca à 10× la vitesse réelle.

GPS_PRODUCER_CODE = '''
#!/usr/bin/env python3
"""
vehicle_gps_producer.py
TaaSim W1 — Producteur GPS simulé
Rejoue les polylines Porto dans Kafka topic raw.gps
à 10× la vitesse réelle avec bruit et blackouts.
"""

import json, time, random, math, argparse
from datetime import datetime, timezone
from kafka import KafkaProducer
import pandas as pd

# ── Config ────────────────────────────────────────────────
KAFKA_BROKER   = "kafka:9092"
TOPIC_GPS      = "raw.gps"
REPLAY_SPEED   = 10          # 10× la vitesse réelle
GPS_INTERVAL   = 15          # 1 ping Porto = 15 secondes réelles
BLACKOUT_PROB  = 0.05        # 5% de chance de blackout par événement
BLACKOUT_DELAY = (60, 180)   # Délai blackout en secondes
GPS_JITTER_DEG = 0.0002      # ±20m de bruit GPS (σ en degrés)

# Bounding boxes
PORTO_LON_MIN, PORTO_LON_MAX = -8.83, -8.43
PORTO_LAT_MIN, PORTO_LAT_MAX =  41.10,  41.25
CASA_LON_MIN,  CASA_LON_MAX  = -7.70, -7.50
CASA_LAT_MIN,  CASA_LAT_MAX  =  33.45,  33.68

def remap_lon(lon):
    r = (lon - PORTO_LON_MIN) / (PORTO_LON_MAX - PORTO_LON_MIN)
    return CASA_LON_MIN + r * (CASA_LON_MAX - CASA_LON_MIN)

def remap_lat(lat):
    r = (lat - PORTO_LAT_MIN) / (PORTO_LAT_MAX - PORTO_LAT_MIN)
    return CASA_LAT_MIN + r * (CASA_LAT_MAX - CASA_LAT_MIN)

def add_noise(coord):
    return coord + random.gauss(0, GPS_JITTER_DEG)

def main():
    parser = argparse.ArgumentParser(description="TaaSim GPS Producer")
    parser.add_argument("--csv",    default="/data/train.csv",  help="Chemin du CSV Porto")
    parser.add_argument("--speed",  type=float, default=REPLAY_SPEED, help="Vitesse de replay")
    parser.add_argument("--sample", type=float, default=0.1,    help="Fraction des taxis")
    args = parser.parse_args()

    print(f"[GPS Producer] Chargement Porto CSV : {args.csv}")
    df = pd.read_csv(args.csv, usecols=["TRIP_ID","TAXI_ID","TIMESTAMP","POLYLINE","MISSING_DATA"])
    df = df[df["MISSING_DATA"] == False].dropna(subset=["POLYLINE"])
    df = df[df["POLYLINE"] != "[]"].sample(frac=args.sample)
    print(f"[GPS Producer] {len(df):,} trajets chargés")

    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BROKER,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        key_serializer=lambda k: str(k).encode("utf-8")
    )

    sleep_between_points = GPS_INTERVAL / args.speed
    total_sent = 0

    while True:  # Boucle infinie — replay en continu
        for _, trip in df.iterrows():
            try:
                polyline = json.loads(trip["POLYLINE"])
            except json.JSONDecodeError:
                continue

            taxi_id = int(trip["TAXI_ID"])
            base_ts = int(time.time())  # Temps réel actuel

            for idx, point in enumerate(polyline):
                lon_porto, lat_porto = point[0], point[1]

                # Remapping + bruit GPS
                lon_casa = add_noise(remap_lon(lon_porto))
                lat_casa = add_noise(remap_lat(lat_porto))

                # Estimation vitesse (km/h) entre points successifs
                speed_kmh = 30.0  # Valeur par défaut
                if idx > 0:
                    prev = polyline[idx - 1]
                    dist = math.sqrt(
                        ((lon_porto - prev[0]) * 111000 * math.cos(math.radians(lat_porto))) ** 2 +
                        ((lat_porto - prev[1]) * 111000) ** 2
                    )
                    speed_kmh = min((dist / 15) * 3.6, 150)  # Cap 150 km/h

                # Timestamp événement (event time) avec décalage éventuel
                event_ts = base_ts + idx * GPS_INTERVAL

                # Simuler blackout : retarder l'envoi Kafka
                blackout_delay = 0
                if random.random() < BLACKOUT_PROB:
                    blackout_delay = random.randint(*BLACKOUT_DELAY)
                    # On envoie quand même mais avec le bon event_time
                    # → Flink reçoit l'événement en retard (test des watermarks)

                event = {
                    "taxi_id":         taxi_id,
                    "event_timestamp": event_ts,          # Event time (Porto time)
                    "ingest_timestamp": int(time.time()), # Processing time
                    "lat":             round(lat_casa, 6),
                    "lon":             round(lon_casa, 6),
                    "speed_kmh":       round(speed_kmh, 1),
                    "status":          "available",
                    "blackout_delay":  blackout_delay
                }

                # Délai de blackout avant envoi
                if blackout_delay > 0:
                    time.sleep(blackout_delay / args.speed)

                producer.send(TOPIC_GPS, key=taxi_id, value=event)
                total_sent += 1

                if total_sent % 1000 == 0:
                    print(f"[GPS] {total_sent:,} événements envoyés | taxi={taxi_id} | "
                          f"lat={lat_casa:.4f} lon={lon_casa:.4f}")

                time.sleep(sleep_between_points)

        print(f"[GPS Producer] Fin du replay — redémarrage ({total_sent:,} total)")

if __name__ == "__main__":
    main()
'''

# Sauvegarder le script
with open("/tmp/vehicle_gps_producer.py", "w") as f:
    f.write(GPS_PRODUCER_CODE)
print("✅ vehicle_gps_producer.py sauvegardé → /tmp/vehicle_gps_producer.py")

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 9d — trip_request_producer.py (code complet)
# ─────────────────────────────────────────────────────────

TRIP_PRODUCER_CODE = '''
#!/usr/bin/env python3
"""
trip_request_producer.py
TaaSim W1 — Producteur de réservations de courses
Génère des événements trip_request sur raw.trips
en respectant la courbe de demande Porto.
"""

import json, time, uuid, random, argparse
from datetime import datetime, timezone
from kafka import KafkaProducer

KAFKA_BROKER = "kafka:9092"
TOPIC_TRIPS  = "raw.trips"

# Poids horaires (générés à l'étape 6 de ce notebook)
# Valeur = fraction de la demande max par heure (0.0 → 1.0)
HOURLY_WEIGHTS = {
    0: 0.12, 1: 0.08, 2: 0.06, 3: 0.05, 4: 0.06, 5: 0.10,
    6: 0.30, 7: 0.75, 8: 1.00, 9: 0.70, 10: 0.55, 11: 0.50,
    12: 0.60, 13: 0.55, 14: 0.50, 15: 0.60, 16: 0.80,
    17: 0.95, 18: 1.00, 19: 0.75, 20: 0.55, 21: 0.40,
    22: 0.28, 23: 0.18
}

# 16 zones Casablanca
ZONES = list(range(1, 17))

# Adjacences (pour le fallback Job 3)
ZONE_ADJACENCY = {
    1: [2, 10, 13], 2: [1, 5, 9],   3: [6, 12, 16],
    4: [9, 13, 15], 5: [2, 9, 14],  6: [3, 16],
    7: [11],        8: [13, 15],    9: [2, 4, 5, 12],
    10: [1, 13],    11: [7, 14],    12: [3, 9, 16],
    13: [1, 4, 8, 10, 15], 14: [5, 11], 15: [4, 8, 13],
    16: [3, 6, 12]
}

# Distribution CALL_TYPE (issue de l'analyse Porto)
CALL_TYPES   = ["A", "B", "C"]
CALL_WEIGHTS = [0.31, 0.16, 0.53]   # Valeurs approximatives Porto

def get_base_rate(hour, day_of_week, base_max=5.0):
    """
    Taux d'émission en événements/seconde.
    base_max = taux max à l'heure de pointe.
    Vendredi (4 = jeudi 0-indexed car day_of_week Spark 1=dimanche)
    → réduction à 12h-14h de 30%.
    """
    weight = HOURLY_WEIGHTS.get(hour, 0.3)
    # Vendredi midi : prière → réduction
    if day_of_week == 6 and 12 <= hour <= 13:   # 6 = vendredi
        weight *= 0.7
    # Week-end : légèrement moins
    if day_of_week in [1, 7]:
        weight *= 0.8
    return max(0.1, base_max * weight)

def generate_trip_event(rider_id):
    origin = random.choice(ZONES)
    # Destination différente de l'origine
    dest = random.choice([z for z in ZONES if z != origin])
    call_type = random.choices(CALL_TYPES, weights=CALL_WEIGHTS)[0]
    
    return {
        "trip_id":       str(uuid.uuid4()),
        "rider_id":      rider_id,
        "origin_zone":   origin,
        "dest_zone":     dest,
        "requested_at":  int(datetime.now(timezone.utc).timestamp()),
        "call_type":     call_type,
        "status":        "pending"
    }

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--max-rate", type=float, default=5.0,
                        help="Événements/sec à l'heure de pointe")
    args = parser.parse_args()

    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BROKER,
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        key_serializer=lambda k: str(k).encode("utf-8")
    )

    total_sent = 0
    print(f"[Trip Producer] Démarrage — taux max : {args.max_rate} req/sec")

    while True:
        now = datetime.now()
        rate = get_base_rate(now.hour, now.isoweekday(), args.max_rate)
        sleep_sec = 1.0 / rate

        rider_id = random.randint(1000, 99999)
        event    = generate_trip_event(rider_id)

        producer.send(TOPIC_TRIPS, key=event["trip_id"], value=event)
        total_sent += 1

        if total_sent % 100 == 0:
            print(f"[Trips] {total_sent:,} | heure={now.hour}h | "
                  f"taux={rate:.2f}/s | zone={event[\'origin_zone\']}→{event[\'dest_zone\']}")

        time.sleep(sleep_sec)

if __name__ == "__main__":
    main()
'''

with open("/tmp/trip_request_producer.py", "w") as f:
    f.write(TRIP_PRODUCER_CODE)
print("✅ trip_request_producer.py sauvegardé → /tmp/trip_request_producer.py")

In [ ]:
# ─────────────────────────────────────────────────────────
# ÉTAPE 9e — Vérification : consommer quelques messages Kafka
# ─────────────────────────────────────────────────────────

# Attendre quelques secondes que les messages soient reçus
time.sleep(3)

consumer = KafkaConsumer(
    "raw.gps",
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="earliest",   # Lire depuis le début
    enable_auto_commit=False,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=5000        # Stop après 5s sans message
)

print("📥 Lecture des messages dans raw.gps (max 10) :")
print("-" * 60)
count = 0
for msg in consumer:
    print(f"  Offset {msg.offset:4d} | Partition {msg.partition} | {msg.value}")
    count += 1
    if count >= 10:
        break

consumer.close()
print(f"\n✅ {count} messages lus dans raw.gps")

---
## ✅ Récapitulatif — Livrables Semaine 1

| Livrable | Fichier | Statut |
|----------|---------|--------|
| Docker stack running | (screenshot à fournir) | ✅ |
| Notebook profiling Porto | W1_Porto_Exploration.ipynb | ✅ |
| Zones remappées | /tmp/zone_mapping.csv | ✅ |
| Carte Casablanca OSM | /tmp/casablanca_zones_map.html | ✅ |
| Topics Kafka créés | raw.gps, raw.trips, processed.* | ✅ |
| GPS producer | /tmp/vehicle_gps_producer.py | ✅ |
| Trip producer | /tmp/trip_request_producer.py | ✅ |
| Dataset curated MinIO | s3a://curated/porto-trips/ | ✅ |

---

## 🚀 Prochaine étape — Semaine 2
- Configurer **Kafka Connect S3 Sink** pour archiver raw.gps → MinIO/kafka-archive/
- Créer les tables **Cassandra** (vehicle_positions, trips, demand_zones)
- Rédiger l'**ADR** (Architecture Decision Record) sur les choix de stockage

---
*TaaSim · ENSA Al Hoceima · 2025–2026*